<a href="https://colab.research.google.com/github/agbizbuz/learning-ai-ds-ml/blob/main/Course_Work/Cust_Feedbak_Summeriser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q --upgrade langchain langchain-groq pydantic pandas

In [ ]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API')

In [ ]:
import pandas as pd

df = pd.read_csv("customer_feedback.csv")

# Combine feedback into one text blob
all_feedback = "\n".join(
    f"- {text}" for text in df["feedback"].astype(str)
)

print(all_feedback[:500])

- The app crashes frequently when uploading photos
- Customer support is very slow to respond
- Great UI but performance is sluggish
- App crashes again after latest update
- Response times from support are unacceptable


In [ ]:
from pydantic import BaseModel, Field
from typing import List

class ComplaintSummary(BaseModel):
    overall_summary: str = Field(description="High-level summary of customer feedback")
    top_complaints: List[str] = Field(description="Top recurring complaints")
    frequency_insights: str = Field(description="Which complaints appear most often and why they matter")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a product analyst summarizing customer feedback for leadership."
    ),
    (
        "human",
        """
Analyze the following customer feedback.

Tasks:
1. Write a concise overall summary.
2. Identify the most common complaints.
3. Explain which complaints appear most frequently and their impact.

Customer feedback:
{feedback}
"""
    )
])

In [ ]:
from langchain_groq import ChatGroq

model = ChatGroq(
    model="llama-3.1-8b-instant",  # stable, supported, non-deprecated
    temperature=0.2
).with_structured_output(ComplaintSummary)

In [ ]:
chain = (
    prompt
    | model
)

In [ ]:
result = chain.invoke({
    "feedback": all_feedback
})

print("=== OVERALL SUMMARY ===")
print(result.overall_summary)

print("\n=== TOP COMPLAINTS ===")
for c in result.top_complaints:
    print("- ", c)

print("\n=== FREQUENCY INSIGHTS ===")
print(result.frequency_insights)

=== OVERALL SUMMARY ===
Customers are experiencing issues with app performance and slow response times from customer support.

=== TOP COMPLAINTS ===
-  App crashes frequently
-  Slow customer support response times

=== FREQUENCY INSIGHTS ===
The most common complaints are related to app performance and customer support, with customers experiencing frequent crashes and slow response times, which are impacting their overall experience and satisfaction with the app.
